In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv
import json
from datetime import datetime
import random
import gradio as gr
load_dotenv()

In [ ]:
openai_key=os.getenv("OPENAI_API_KEY")   
client=OpenAI(api_key=openai_key)

In [ ]:
def get_weather(city):
    return f"Sunny, 75°F in {city}"
def current_time():
    return f"The time is {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
def tell_joke():
    jokes = [
        "Why don't scientists trust atoms? Because they make up everything!",
        "Why did the scarecrow win an award? Because he was outstanding in his field!",
        "Why do programmers prefer dark mode? Because light attracts bugs!"
    ]
    return random.choice(jokes)
    

In [ ]:


tools=[
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather in a given city",
            "parameters": {
                "type": "object",
                "properties": {'city':{'type':'string','description':'The city to get the weather for'}},
                "required": ['city']

                
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "current_time",
            "description": "Get the current time in a given city",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []

                
            }
        }
    },

    {
        "type": "function",
        "function": {
            "name": "tell_joke",
            "description": "Tell a joke",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []

                
            }
        }
    }
]

    # add current message
def chat(message, history):
    messages=[ {'role':'system','content':'''You are a personal journal assistant.
    Follow these rules strictly:
    - Use the tool best suitable for the task given by user
    - Respond in 2-3 sentences maximum
    - When asked to pick one option, state the choice and one reason only
    - Never use bullet points or numbered lists
    - Never end with phrases like 'feel free to ask' or 'have a great day'''},
    {'role':'assistant','content':'Hi! How can I help you Today?'}]
    
    
    # rebuild history
    for user_msg, assistant_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": assistant_msg})

    messages.append({"role": "user", "content": message})
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools
    )

    if response.choices[0].finish_reason == "tool_calls":
            tool_call = response.choices[0].message.tool_calls[0]
            tool_name = tool_call.function.name
            
            messages.append(response.choices[0].message)  # ← append assistant
            
            if tool_name == "current_time":
                result = current_time()
            elif tool_name == "get_weather":
                args = json.loads(tool_call.function.arguments)
                result = get_weather(args["city"])
            elif tool_name == "tell_joke":
                result = tell_joke()
            
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })
            
            # second API call for final answer
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages
            )
        
    return response.choices[0].message.content


# ── Launch ──
gr.ChatInterface(
    fn=chat,
    title="Personal Assistant",
    description="Ask me about time, weather, or jokes!"
).launch()
      

In [ ]:



#tools=[current_time,get_weather,tell_joke]



In [ ]:

messages.append({"role": "user", "content": message})
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=tools
)

if response.choices[0].finish_reason == "tool_calls":
        tool_call = response.choices[0].message.tool_calls[0]
        tool_name = tool_call.function.name
        
        messages.append(response.choices[0].message)  # ← append assistant
        
        if tool_name == "current_time":
            result = current_time()
        elif tool_name == "get_weather":
            args = json.loads(tool_call.function.arguments)
            result = get_weather(args["city"])
        elif tool_name == "tell_joke":
            result = tell_joke()
        
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result
        })
        
        # second API call for final answer
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages
        )
    
return response.choices[0].message.content


# ── Launch ──
gr.ChatInterface(
    fn=chat,
    title="Personal Assistant",
    description="Ask me about time, weather, or jokes!"
).launch()